# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

In [ ]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [3]:

import pandas as pd

df = pd.read_csv("orders.csv")

print("Данные загружены!")
print(df.head())

print("Размер:", df.shape)

print("Типы столбцов:\n", df.dtypes)

print("Пропусков delivery_days:",
      df["delivery_days"].isna().sum())

for column in ["city", "category", "status", "customer_id"]:
    print(column, "уникальных значений:",
          df[column].nunique())


Данные загружены!
   order_id        order_date customer_id       city category   amount  \
0    101674  2026-09-01 08:00      C00681     Алматы   Одежда   8070.0   
1    101214  2026-09-01 08:00      C01210     Астана  Красота   3630.0   
2    102424  2026-09-01 08:00      C01408  Караганда      Дом   9120.0   
3    102333  2026-09-01 08:00      C01529     Алматы   Одежда   9350.0   
4    102564  2026-09-01 09:00      C00657     Астана      Дом  11010.0   

   delivery_days   status  
0            1.0  оплачен  
1            2.0  оплачен  
2            3.0  оплачен  
3            2.0  оплачен  
4            1.0  оплачен  
Размер: (5463, 8)
Типы столбцов:
 order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount               str
delivery_days    float64
status               str
dtype: object
Пропусков delivery_days: 160
city уникальных значений: 14
category уникальных значений: 5
status уникальных значений: 2
c

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [6]:

# 2. Типичный чек: оплаченные и без TEST

# Превращаем текст в числа
df["amount"] = pd.to_numeric(
    df["amount"].astype("string").str.replace(" ", "", regex=False),
    errors="coerce"
)

# Создаём маски
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"

# Отбираем нужные заказы
checks = df.loc[paid & not_test, "amount"]

# Вычисляем
print("Заказов:", len(checks))
print("Среднее:", round(checks.mean(), 2), "₸")
print("Медиана:", checks.median(), "₸")


Заказов: 4901
Среднее: 14672.39 ₸
Медиана: 10300.0 ₸


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [7]:
# 3
# 3. Отмены по городам

# Все варианты написания городов
SH = ["Шымкент", "шымкент", "Shymkent"]
AL = ["Алматы", "алматы", "АЛМАТЫ", "Almaty", " Алматы"]

# Создаём маски городов
sh_mask = df["city"].isin(SH)
al_mask = df["city"].isin(AL)

# Маска отменённых заказов
cancel = df["status"] == "отменён"

# Рассчитываем доли отмен
rate_all = cancel.mean()
rate_sh = (cancel & sh_mask).sum() / sh_mask.sum()
rate_al = (cancel & al_mask).sum() / al_mask.sum()

# Выводим результаты
print(f"Все города: {rate_all:.2%}")
print(f"Шымкент: {rate_sh:.2%}")
print(f"Алматы: {rate_al:.2%}")

print(f"Шымкент / Алматы: {rate_sh / rate_al:.2f} раза")


Все города: 10.23%
Шымкент: 20.40%
Алматы: 7.86%
Шымкент / Алматы: 2.59 раза


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [8]:

# 4. Выручка по городам: сентябрь и октябрь

SH = ["Шымкент", "шымкент", "Shymkent"]
AL = ["Алматы", "алматы", "АЛМАТЫ", "Almaty", " Алматы"]

# Маски
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"

sep = df["order_date"].str[:7] == "2026-09"
octo = df["order_date"].str[:7] == "2026-10"

# Считаем изменения
changes = {}

for name, spellings in [("Шымкент", SH), ("Алматы", AL)]:
    city_mask = df["city"].isin(spellings)

    september = df.loc[
        city_mask & paid & not_test & sep, "amount"
    ].sum()

    october = df.loc[
        city_mask & paid & not_test & octo, "amount"
    ].sum()

    change_pct = (october / september - 1) * 100

    changes[name] = change_pct

    print(
        f"{name}: сентябрь {september:,.0f} ₸, "
        f"октябрь {october:,.0f} ₸; "
        f"{change_pct:.2f}%"
    )

ratio = abs(changes["Шымкент"] / changes["Алматы"])

print(f"Падение Шымкента сильнее в {ratio:.2f} раза.")

print(
    f"Вывод менеджеру: изучить Шымкент — "
    f"изменение {changes['Шымкент']:.2f}% "
    f"против {changes['Алматы']:.2f}% в Алматы."
)


Шымкент: сентябрь 4,949,800 ₸, октябрь 4,044,010 ₸; -18.30%
Алматы: сентябрь 15,523,940 ₸, октябрь 14,643,020 ₸; -5.67%
Падение Шымкента сильнее в 3.22 раза.
Вывод менеджеру: изучить Шымкент — изменение -18.30% против -5.67% в Алматы.


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [9]:

# 5. Исправляем две ошибки AI

# Все варианты написания Шымкента
SH = ["Шымкент", "шымкент", "Shymkent"]

# Выбираем отменённые заказы Шымкента
cancel_sh = df[
    (df["city"].isin(SH)) &
    (df["status"] == "отменён")
]

# Выводим результат
print("Отменённых заказов Шымкента:", len(cancel_sh))

print(
    f"Вывод менеджеру: {len(cancel_sh)} отменённых заказов. "
    "Нужно учитывать все три написания города."
)


Отменённых заказов Шымкента: 164
Вывод менеджеру: 164 отменённых заказов. Нужно учитывать все три написания города.


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [10]:

# 6. Самый частый клиент и повторяющиеся order_id

import pandas as pd

df = pd.read_csv("orders.csv")

# Считаем заказы каждого клиента
counts = df["customer_id"].value_counts()

# Находим самого частого клиента
top_customer = counts.index[0]

# Выбираем все его заказы
client_orders = df.loc[
    df["customer_id"] == top_customer,
    ["order_date", "city", "amount", "status", "order_id"]
]

# Считаем повторяющиеся order_id
client_duplicates = client_orders["order_id"].duplicated().sum()
all_duplicates = df["order_id"].duplicated().sum()

# Вывод результата
print("Клиент:", top_customer)
print("Количество строк:", counts.iloc[0])

print(client_orders)

print("Дубли клиента:", client_duplicates)
print("Дубли всей таблицы:", all_duplicates)

print(
    f"Вывод менеджеру: у клиента {top_customer} "
    f"обнаружено {client_duplicates} повторов order_id. "
    f"Во всей таблице {all_duplicates} повторов."
)


Клиент: C01818
Количество строк: 11
            order_date       city   amount   status  order_id
57    2026-09-01 15:00    Шымкент   7350.0  оплачен    101125
779   2026-09-09 18:00    Шымкент  45270.0  оплачен    100096
1371  2026-09-15 21:00     Алматы   8890.0  оплачен    101504
1375  2026-09-15 21:00     Алматы   8890.0  оплачен    101504
2195  2026-09-24 18:00     Астана   1490.0  оплачен    101860
2617  2026-09-29 09:00     Астана  22170.0  оплачен    101462
2776  2026-09-30 21:00     Алматы   20 180  оплачен    101495
3417  2026-10-08 14:00  Караганда   4330.0  оплачен    103416
4279  2026-10-18 08:00     Астана  13600.0  оплачен    102644
4610  2026-10-21 19:00     Алматы  21990.0  оплачен    104001
4778  2026-10-23 18:00     Алматы  11140.0  оплачен    104572
Дубли клиента: 1
Дубли всей таблицы: 180
Вывод менеджеру: у клиента C01818 обнаружено 1 повторов order_id. Во всей таблице 180 повторов.


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8